# Summarization quality benchmark

Generate summaries from `mteb/summeval`, then score them with BERTScore F1 and HHEM.

Workflow: configure, prepare checkpoints, generate, score, compare. Scoring uses saved summaries without calling the generator.

Use the project Jupyter kernel and start MLflow with `docker compose up -d mlflow`. Generation may incur charges; scorers download weights on first use. Human references and ratings are never sent to the generator.


In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

import gc
import platform
import random
import re
import warnings
import traceback
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from time import perf_counter
from uuid import uuid4

import mlflow
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from tqdm.auto import tqdm

from utils.summarization_quality import (
    GenerationResponseError, compare_generations, fingerprint, generate_articles, generation_records,
    load_model_configs, prepare_generation, publish_results, read_json, score_articles,
    score_records, write_json,
)

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter in the project root or a subfolder.")
load_dotenv(PROJECT_ROOT / ".env")
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "summarization-quality"


def package_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = None
    return result


def publish_safely(directory, metric_spec=None, *, interrupted=False):
    try:
        run_id = publish_results(directory, metric_spec, interrupted=interrupted)
        print(f"MLflow: {run_id}")
    except Exception as error:
        warnings.warn(
            f"MLflow upload failed: {error}. Local checkpoints are safe; "
            "rerun the publication cell when the server is available."
        )


## Configuration

Edit `benchmark/model-configs.json`: `defaults` holds shared settings, model entries override them, and `enabled=false` skips a model. Rerun this cell after edits. API keys stay in `.env`. Ollama entries need a matching Hugging Face tokenizer.

`TESTING=True` selects 5 articles with a fixed seed; full mode uses all articles. `REPEATS` controls independent generations. Models receive the same articles and prompt requesting 2 to 3 English sentences. Calls are sequential without automatic retries. The 2048-token output budget includes reasoning.

Set `GENERATION_DIRS` to resume checkpoints using their saved settings. For a new batch, clear the list and disable `USE_EXISTING_GENERATIONS`. Model configuration changes apply only to new batches.


In [2]:
TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "summarization-quality-benchmark"
BENCHMARK_GROUP = "summeval-v1"
TESTING = os.getenv("SUMMARIZATION_TESTING", "false").lower() == "true"
SAMPLE_SEED = 42
REPEATS = 1
DATASET_NAME = "mteb/summeval"
DATASET_REVISION = "main"  # Resolved to a commit when preparing a new batch.
DATASET_SPLIT = "test"
USE_EXISTING_GENERATIONS = True

SYSTEM_PROMPT = (
    "You summarize English news articles accurately and concisely. "
    "Use only facts supported by the article. Return only the summary."
)
USER_PROMPT = "Summarize the article in 2 to 3 concise English sentences.\n\nArticle:\n{text}"

MODEL_CONFIG_PATH = PROJECT_ROOT / "benchmark" / "model-configs.json"
MODEL_CONFIGS = load_model_configs(MODEL_CONFIG_PATH)

# Explicit paths take priority over automatic selection.
GENERATION_DIRS = [
    # ARTIFACT_ROOT / "51044cf5f89a4309a0f2e6da9aecde2e",
]
mlflow.set_tracking_uri(TRACKING_URI)

display(pd.DataFrame(MODEL_CONFIGS))


,base_url,num_ctx,num_gpu,reasoning,top_k,repeat_penalty,tokenizer_revision,temperature,max_output_tokens,timeout_seconds,provider,model,tokenizer
0,http://127.0.0.1:11434,65536,0,False,50,1.1,main,0,8192,1800,ollama,openbmb/minicpm5-2b,openbmb/MiniCPM5-2B


## Prepare checkpoints

New batches save a frozen dataset, pinned revisions, and a manifest per model/repetition. Existing checkpoints reuse their snapshots without downloads. This cell makes no generation calls and keeps the selected directories when rerun.

Local JSON files support resume, MLflow stores copies. Use one kernel per directory.


In [3]:
if USE_EXISTING_GENERATIONS and not GENERATION_DIRS:
    for path in sorted(ARTIFACT_ROOT.glob("*/manifest.json")):
        saved = read_json(path)
        if (
            saved["benchmark_group"] == BENCHMARK_GROUP
            and saved["experiment_name"] == EXPERIMENT_NAME
            and saved["protocol"]["dataset"]["testing"] == TESTING
        ):
            GENERATION_DIRS.append(path.parent)
    if not GENERATION_DIRS:
        raise RuntimeError(
            "No saved generations match BENCHMARK_GROUP, EXPERIMENT_NAME, and TESTING. "
            "Check those filters or provide GENERATION_DIRS explicitly."
        )

if GENERATION_DIRS:
    GENERATION_DIRS = [Path(path) for path in GENERATION_DIRS]
    for directory in GENERATION_DIRS:
        manifest = read_json(directory / "manifest.json")
        prepare_generation(directory, manifest)
        print(f"Existing: {directory} | {manifest['generator']['model']}")
else:
    from datasets import load_dataset
    from huggingface_hub import HfApi

    if REPEATS < 1 or not MODEL_CONFIGS:
        raise ValueError("Provide models and set REPEATS >= 1.")
    if not BENCHMARK_GROUP.strip():
        raise ValueError("BENCHMARK_GROUP must not be empty.")
    for config in MODEL_CONFIGS:
        if config["provider"] not in {"ollama", "openrouter"} or not config["model"].strip():
            raise ValueError("Each configuration needs a supported provider and model.")
        if config["max_output_tokens"] <= 0 or config["timeout_seconds"] <= 0:
            raise ValueError("Token limit and timeout must be positive.")
        if config["provider"] == "ollama" and config["num_ctx"] <= config["max_output_tokens"]:
            raise ValueError("num_ctx must leave space for input.")

    api = HfApi()
    dataset_revision = api.dataset_info(DATASET_NAME, revision=DATASET_REVISION).sha
    dataset = load_dataset(DATASET_NAME, revision=dataset_revision, split=DATASET_SPLIT)
    indices = (
        sorted(random.Random(SAMPLE_SEED).sample(range(len(dataset)), min(5, len(dataset))))
        if TESTING else list(range(len(dataset)))
    )
    articles = [dict(dataset[index]) for index in indices]
    dataset_spec = {
        "name": DATASET_NAME, "revision": dataset_revision, "split": DATASET_SPLIT,
        "testing": TESTING, "sample_seed": SAMPLE_SEED,
        "article_ids": [article["id"] for article in articles],
        "sha256": fingerprint(articles),
    }
    prompt = {"system": SYSTEM_PROMPT, "user_template": USER_PROMPT}
    protocol = {
        "dataset": dataset_spec, "prompt": prompt,
        "preprocessing_version": 1, "summary_length_policy": "2_to_3_sentences",
    }
    environment = {
        "python": platform.python_version(), "platform": platform.platform(),
        "host": platform.node(),
        "packages": package_versions([
            "datasets", "langchain-ollama", "langchain-openrouter",
            "langchain-core", "transformers", "huggingface-hub", "mlflow",
        ]),
    }
    configs = []
    for original in MODEL_CONFIGS:
        config = dict(original)
        if config["provider"] == "ollama":
            config["tokenizer_revision"] = api.model_info(
                config["tokenizer"], revision=config["tokenizer_revision"],
            ).sha
        configs.append(config)

    batch_id = uuid4().hex
    for repeat_index in range(1, REPEATS + 1):
        for config in configs:
            generation_id = uuid4().hex
            manifest = {
                "schema_version": 1, "generation_id": generation_id,
                "batch_id": batch_id, "benchmark_group": BENCHMARK_GROUP,
                "experiment_name": EXPERIMENT_NAME, "repeat_index": repeat_index,
                "protocol_sha256": fingerprint(protocol), "protocol": protocol,
                "generator": config, "client_retries": 0, "concurrency": 1,
                "environment": environment, "articles": articles,
            }
            directory = prepare_generation(ARTIFACT_ROOT / generation_id, manifest)
            GENERATION_DIRS.append(directory)
    write_json(
        ARTIFACT_ROOT / "batches" / f"{batch_id}.json",
        {"generation_directories": [str(path) for path in GENERATION_DIRS]},
    )

display(pd.DataFrame([
    {
        "directory": str(path),
        "model": read_json(path / "manifest.json")["generator"]["model"],
        "articles": len(read_json(path / "manifest.json")["articles"]),
        "completed": sum(row["status"] == "ok" for row in generation_records(path)),
    }
    for path in GENERATION_DIRS
]))


Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\76acfbf8b8414e908bf5f38b56eb5315 | openbmb/minicpm5-2b
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\7e8caf67136c4d5a87dbbef4475680b3 | maternion/ling-3.0-tiny:8b-Q4_K_M
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\823942191bd74154a8b0d7c7a52e3752 | openbmb/minicpm5-2b
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\928dcb87506246379f763a99f7b968b1 | gpt-6-luna
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\a9fdec5619124b0fb8a032abd059e495 | gpt-6-luna
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\c453a4246c324e03afce0ca112efb1cd | qwen3.5:4b-q4_K_M
Existing: d:\Workspace\Repository\llm-summarization\artifacts\summarization-quality\c911ff3b41b4465eb778d215d59b64e9 | LiquidAI/lfm2.5-2.6b:q4_k_m
Existing: d:\Workspace\Re

,directory,model,articles,completed
0,d:\Workspace\Repository\llm-summarization\arti...,openbmb/minicpm5-2b,100,0
1,d:\Workspace\Repository\llm-summarization\arti...,maternion/ling-3.0-tiny:8b-Q4_K_M,100,100
2,d:\Workspace\Repository\llm-summarization\arti...,openbmb/minicpm5-2b,100,100
3,d:\Workspace\Repository\llm-summarization\arti...,gpt-6-luna,100,100
4,d:\Workspace\Repository\llm-summarization\arti...,gpt-6-luna,100,100
5,d:\Workspace\Repository\llm-summarization\arti...,qwen3.5:4b-q4_K_M,100,100
6,d:\Workspace\Repository\llm-summarization\arti...,LiquidAI/lfm2.5-2.6b:q4_k_m,100,100
7,d:\Workspace\Repository\llm-summarization\arti...,granite4.2:3b-q4_K_M,100,100
8,d:\Workspace\Repository\llm-summarization\arti...,gpt-6-luna,100,100
9,d:\Workspace\Repository\llm-summarization\arti...,gpt-6-luna,100,100


In [4]:
def build_generator(manifest):
    from langchain_core.messages import HumanMessage, SystemMessage

    config = manifest["generator"]
    prompt = manifest["protocol"]["prompt"]
    tokenizer = None
    if config["provider"] == "ollama":
        from langchain_ollama import ChatOllama
        from transformers import AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(
            config["tokenizer"], revision=config["tokenizer_revision"],
        )
        model = ChatOllama(
            model=config["model"], temperature=config["temperature"],
            base_url=config["base_url"], validate_model_on_init=True,
            num_predict=config["max_output_tokens"], num_ctx=config["num_ctx"],
            num_gpu=config["num_gpu"], reasoning=config["reasoning"],
            top_k=config["top_k"], repeat_penalty=config["repeat_penalty"],
            client_kwargs={"timeout": config["timeout_seconds"]},
        )
    else:
        from langchain_openrouter import ChatOpenRouter

        if not os.getenv("OPENROUTER_API_KEY"):
            raise ValueError("Set OPENROUTER_API_KEY in .env.")
        model = ChatOpenRouter(
            model=config["model"], temperature=config["temperature"],
            max_tokens=config["max_output_tokens"], max_retries=0,
            timeout=int(config["timeout_seconds"] * 1000),
        )

    def generate(article):
        user_text = prompt["user_template"].format(text=article["text"])
        if tokenizer is not None:
            chat = [
                {"role": "system", "content": prompt["system"]},
                {"role": "user", "content": user_text},
            ]
            if tokenizer.chat_template:
                input_tokens = len(tokenizer.apply_chat_template(
                    chat, tokenize=True, add_generation_prompt=True,
                ))
            else:
                input_tokens = len(tokenizer.encode(prompt["system"] + "\n\n" + user_text))
            if input_tokens + config["max_output_tokens"] > config["num_ctx"]:
                raise ValueError("Input plus output budget exceeds num_ctx.")

        started = perf_counter()
        response = model.invoke([
            SystemMessage(content=prompt["system"]), HumanMessage(content=user_text),
        ])
        request_seconds = perf_counter() - started
        raw = response.content
        text = raw if isinstance(raw, str) else "".join(
            block if isinstance(block, str) else block.get("text", "")
            for block in raw
            if isinstance(block, str) or block.get("type") in {"text", "output_text"}
        )
        metadata = response.response_metadata or {}
        usage = response.usage_metadata or {}
        details = {
            "raw_content": raw,
            "response_metadata": metadata,
            "usage": usage,
            "token_count_source": "provider" if usage else "unavailable",
            "finish_reason": metadata.get("finish_reason") or metadata.get("done_reason"),
            "request_seconds": request_seconds,
        }
        summary = re.sub(r"<think>.*?</think>", "", text, flags=re.I | re.S).strip()
        # Never evaluate unfinished reasoning as if it were a final answer.
        if re.search(r"<think>", summary, re.I):
            raise GenerationResponseError(
                "Response has an incomplete reasoning block. "
                f"finish_reason={details['finish_reason']}; "
                f"output_tokens={usage.get('output_tokens')}; "
                f"max_output_tokens={config['max_output_tokens']}. "
                "Reasoning can consume the output budget even when reasoning=False. "
                "Increase max_output_tokens in a NEW generation configuration.",
                details,
            )
        # Some templates prefill the opening tag, so only the closing tag is returned.
        if re.search(r"</think>", summary, re.I):
            summary = re.split(r"</think>", summary, flags=re.I)[-1].strip()
        if not summary:
            raise GenerationResponseError(
                "Model returned no final summary. "
                f"finish_reason={details['finish_reason']}; "
                f"max_output_tokens={config['max_output_tokens']}.",
                details,
            )
        details["summary_words"] = len(summary.split())
        return summary, details

    return generate


## Generate summaries

Run to generate or resume. Completed articles are skipped, failed or missing articles are retried. Each new client makes a sanity request outside benchmark measurements.

Progress is saved per article, including failures and truncated outputs. Rerun after an interruption to continue.


In [5]:
if not GENERATION_DIRS:
    raise RuntimeError("Prepare or select generation directories first.")

for directory in tqdm(GENERATION_DIRS, desc="Generation runs", unit="run"):
    manifest = read_json(directory / "manifest.json")
    done = sum(row["status"] == "ok" for row in generation_records(directory))
    interrupted = False
    try:
        if done < len(manifest["articles"]):
            generate = build_generator(manifest)
            sanity, _ = generate({"id": "sanity", "text": "The meeting is on Monday."})
            if not sanity.strip():
                raise ValueError("Sanity check returned an empty summary.")
            print(f"Sanity ({manifest['generator']['model']}): {sanity}")
            generate_articles(
                directory, generate,
                progress=lambda rows: tqdm(rows, desc="Articles", unit="article", leave=False),
            )
            del generate
    except KeyboardInterrupt:
        interrupted = True
        raise
    except Exception as error:
        write_json(directory / "setup_error.json", {
            "error": traceback.format_exc(),
            "metadata": getattr(error, "metadata", None),
        })
        warnings.warn(f"Generation setup failed for {directory.name}: {error}")
    finally:
        publish_safely(directory, interrupted=interrupted)


Generation runs:   0%|          | 0/10 [00:00<?, ?run/s]

d:\Workspace\Repository\llm-summarization\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


Sanity (openbmb/minicpm5-2b): I'm sorry, I can't summarize an article that's too short or doesn't contain enough information for a meaningful summary. Please provide more details about the content you'd like me to work with.


Articles:   0%|          | 0/100 [00:00<?, ?article/s]

C:\Users\Andakara\AppData\Local\Temp\ipykernel_215028\1430523343.py:53: UserWarning: MLflow upload failed: Cannot start run with ID 63147efb75054c79afed30ac1f859355 because it is in the deleted state.. Local checkpoints are safe; rerun the publication cell when the server is available.
  warnings.warn(


🏃 View run generation: maternion/ling-3.0-tiny:8b-Q4_K_M (7e8caf67) at: http://127.0.0.1:5000/#/experiments/2/runs/cbba334ea7874f278beaad68d7ecfa0c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run generation: maternion/ling-3.0-tiny:8b-Q4_K_M (7e8caf67) at: http://127.0.0.1:5000/#/experiments/2/runs/cbba334ea7874f278beaad68d7ecfa0c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: cbba334ea7874f278beaad68d7ecfa0c
🏃 View run generation: openbmb/minicpm5-2b (82394219) at: http://127.0.0.1:5000/#/experiments/2/runs/f59dbf54c5734d61bee9dc8cceb10d33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run generation: openbmb/minicpm5-2b (82394219) at: http://127.0.0.1:5000/#/experiments/2/runs/f59dbf54c5734d61bee9dc8cceb10d33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: f59dbf54c5734d61bee9dc8cceb10d33


C:\Users\Andakara\AppData\Local\Temp\ipykernel_215028\1430523343.py:53: UserWarning: MLflow upload failed: Cannot start run with ID 4d4d8a36cdf842248e11ac7769a4f768 because it is in the deleted state.. Local checkpoints are safe; rerun the publication cell when the server is available.
  warnings.warn(


🏃 View run generation: gpt-6-luna (a9fdec56) at: http://127.0.0.1:5000/#/experiments/2/runs/6ef318e2a4814ba98591ea9b4b9a4b11
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run generation: gpt-6-luna (a9fdec56) at: http://127.0.0.1:5000/#/experiments/2/runs/6ef318e2a4814ba98591ea9b4b9a4b11
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: 6ef318e2a4814ba98591ea9b4b9a4b11
🏃 View run generation: qwen3.5:4b-q4_K_M (c453a424) at: http://127.0.0.1:5000/#/experiments/2/runs/d21221e60d704092a1dc4cc6fa87a0ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run generation: qwen3.5:4b-q4_K_M (c453a424) at: http://127.0.0.1:5000/#/experiments/2/runs/d21221e60d704092a1dc4cc6fa87a0ca
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: d21221e60d704092a1dc4cc6fa87a0ca
🏃 View run generation: LiquidAI/lfm2.5-2.6b:q4_k_m (c911ff3b) at: http://127.0.0.1:5000/#/experiments/2/runs/1fd5514f45334c4e817ec9e78cfd06fb
🧪 View experiment at: 

## Configure metrics

Select `METRIC_NAMES`, or reuse saved `scores/<metric-id>/metric.json` files via `METRIC_SPEC_PATHS`. This cell saves scorer settings and revisions without loading models.

- **BERTScore F1:** DeBERTa-xlarge-MNLI layer 40, no IDF, baseline rescaling, slow tokenizer, and best F1 across human references. Scores can be negative. 
- **Vectera HHEM-2.1-Open (Cosistency):** measures source support using the article and generated summary. Higher is better.

The default device is `cuda`. Set `SCORER_DEVICE="cpu"` for CPU, or reduce `BERTSCORE_BATCH_SIZE` if GPU memory is insufficient. Reused specs keep their saved settings.

Sources: [BERTScore](https://github.com/Tiiiger/bert_score), [Vectera HHEM-2.1-Open](https://huggingface.co/vectara/hallucination_evaluation_model).


In [12]:
def validate_scorer_device(value):
    if not isinstance(value, str) or not value.strip() or value != value.strip():
        raise ValueError(
            f"Invalid scorer device {value!r}. Use 'cpu' or 'cuda', without spaces."
        )
    import torch

    try:
        device = torch.device(value)
    except (RuntimeError, ValueError, TypeError) as error:
        raise ValueError(f"Invalid scorer device {value!r}. Use 'cpu' or 'cuda'.") from error
    if device.type == "cuda":
        if not torch.cuda.is_available():
            raise ValueError("CUDA is unavailable in this environment; use SCORER_DEVICE='cpu'.")
        if device.index is not None and device.index >= torch.cuda.device_count():
            raise ValueError(f"CUDA device {device.index} is unavailable.")


METRIC_SPEC_PATHS = [
    # ARTIFACT_ROOT / "generation-id/scores/metric-id/metric.json",
]
# METRIC_NAMES = ["bertscore_f1", "hhem_consistency"]  
# SCORER_DEVICE = os.getenv("SUMMARIZATION_SCORER_DEVICE", "cuda")
METRIC_NAMES = ["hhem_consistency"]
SCORER_DEVICE = "cuda"
BERTSCORE_BATCH_SIZE = 4
BERTSCORE_MODEL = "microsoft/deberta-xlarge-mnli"
if not METRIC_NAMES or set(METRIC_NAMES) - {"bertscore_f1", "hhem_consistency"}:
    raise ValueError("Choose bertscore_f1 and/or hhem_consistency in METRIC_NAMES.")
BERTSCORE_REVISION = "main"
HHEM_REVISION = "main"
HHEM_FOUNDATION_REVISION = "main"

if METRIC_SPEC_PATHS:
    METRIC_SPECS = [
        spec for path in METRIC_SPEC_PATHS
        if (spec := read_json(Path(path)))["name"] in METRIC_NAMES
    ]
else:
    validate_scorer_device(SCORER_DEVICE)
    from huggingface_hub import HfApi

    api = HfApi()
    metric_environment = package_versions([
        "bert-score", "transformers", "torch", "tokenizers",
        "huggingface-hub", "sentencepiece",
    ])
    METRIC_SPECS = []
    if "bertscore_f1" in METRIC_NAMES:
        METRIC_SPECS.append({
            "name": "bertscore_f1", "implementation_version": 1,
            "model": BERTSCORE_MODEL,
            "revision": api.model_info(BERTSCORE_MODEL, revision=BERTSCORE_REVISION).sha,
            "num_layers": 40, "lang": "en", "idf": False,
            "rescale_with_baseline": False, "use_fast_tokenizer": False,
            "reference_aggregation": "max_f1", "overlength_policy": "error",
            "batch_size": BERTSCORE_BATCH_SIZE, "device": SCORER_DEVICE,
            "packages": metric_environment,
        })
    if "hhem_consistency" in METRIC_NAMES:
        METRIC_SPECS.append({
            "name": "hhem_consistency", "implementation_version": 1,
            "model": "vectara/hallucination_evaluation_model",
            "revision": api.model_info(
                "vectara/hallucination_evaluation_model", revision=HHEM_REVISION,
            ).sha,
            "foundation": "google/flan-t5-base",
            "foundation_revision": api.model_info(
                "google/flan-t5-base", revision=HHEM_FOUNDATION_REVISION,
            ).sha,
            "input_order": ["source", "summary"], "truncation": False,
            "device": SCORER_DEVICE, "packages": metric_environment,
        })
if not METRIC_SPECS:
    raise ValueError("No metric specs match METRIC_NAMES. Check METRIC_SPEC_PATHS.")
for spec in METRIC_SPECS:
    validate_scorer_device(spec["device"])
    # A requested spec must describe the actual installed implementation.
    if spec["packages"] != package_versions(spec["packages"]):
        raise ValueError("Scorer package versions differ from the saved metric spec.")
    write_json(ARTIFACT_ROOT / "metric-specs" / f"{fingerprint(spec)}.json", spec)
display(pd.DataFrame([
    {"metric": spec["name"], "metric_id": fingerprint(spec), "revision": spec["revision"],
     "model": spec["model"], "device": spec["device"]}
    for spec in METRIC_SPECS
]))


,metric,metric_id,revision,model,device
0,hhem_consistency,be20d08c9b2cf78940781184f1c645f3415face31e16da...,de9a5e6ba9ba49531a022dba17e35ffeb2a6fad8,vectara/hallucination_evaluation_model,cuda


In [13]:
def build_evaluator(spec):
    validate_scorer_device(spec["device"])
    from huggingface_hub import snapshot_download

    if spec["implementation_version"] != 1:
        raise ValueError("Unsupported metric implementation version.")
    if spec["name"] == "bertscore_f1":
        if spec["reference_aggregation"] != "max_f1" or spec["overlength_policy"] != "error":
            raise ValueError("Unsupported BERTScore aggregation/overlength policy.")
        from bert_score import BERTScorer
        from transformers import AutoTokenizer

        model_path = snapshot_download(
            spec["model"], revision=spec["revision"],
            allow_patterns=["*.json", "*.safetensors", "*.bin", "vocab.*", "merges.txt"],
        )
        # The model uses a pinned local snapshot, so resolve its baseline explicitly.
        baseline_path = None
        if spec["rescale_with_baseline"]:
            import bert_score

            baseline_path = (
                Path(bert_score.__file__).resolve().parent
                / "rescale_baseline" / spec["lang"] / f"{spec['model']}.tsv"
            )
            if not baseline_path.is_file():
                raise ValueError(f"BERTScore baseline not found: {baseline_path}")
        scorer = BERTScorer(
            model_type=model_path, num_layers=spec["num_layers"], lang=spec["lang"],
            idf=spec["idf"], rescale_with_baseline=spec["rescale_with_baseline"],
            baseline_path=str(baseline_path) if baseline_path is not None else None,
            use_fast_tokenizer=spec["use_fast_tokenizer"], device=spec["device"],
        )
        tokenizer = AutoTokenizer.from_pretrained(
            model_path, use_fast=spec["use_fast_tokenizer"],
        )
        
        scorer._tokenizer.model_max_length = 512
        tokenizer.model_max_length = 512

        def evaluate(article, summary):
            texts = [summary, *article["human_summaries"]]
            lengths = [len(tokenizer.encode(text, truncation=False)) for text in texts]
            if max(lengths) > tokenizer.model_max_length:
                raise ValueError("BERTScore input exceeds encoder token limit.")
            precision, recall, f1 = scorer.score(
                [summary], [article["human_summaries"]],
                batch_size=spec["batch_size"],
            )
            return {
                "value": f1.item(),
                "precision": precision.item(), "recall": recall.item(),
                "bertscore_hash": scorer.hash,
            }

        return evaluate

    if spec["name"] == "hhem_consistency":
        if spec["input_order"] != ["source", "summary"] or spec["truncation"]:
            raise ValueError("HHEM requires source/summary order without truncation.")
        from transformers import AutoConfig, AutoModelForSequenceClassification

        # HHEM's custom loader initializes its foundation tokenizer/config itself.
        # Point it at a pinned local snapshot, not an unversioned Hub name.
        foundation_path = snapshot_download(
            spec["foundation"], revision=spec["foundation_revision"],
            allow_patterns=["*.json", "spiece.model", "tokenizer.model"],
        )
        config = AutoConfig.from_pretrained(
            spec["model"], revision=spec["revision"], code_revision=spec["revision"],
            trust_remote_code=True,
        )
        config.foundation = foundation_path
        scorer = AutoModelForSequenceClassification.from_pretrained(
            spec["model"], revision=spec["revision"], code_revision=spec["revision"],
            config=config, trust_remote_code=True,
        ).to(spec["device"])
        scorer.eval()

        def evaluate(article, summary):
            return scorer.predict([(article["text"], summary)]).item()

        return evaluate

    raise ValueError(f"No evaluator registered for {spec['name']!r}.")


## Score saved summaries

Scores successful summaries and skips cached results. Scorers load only when needed, one metric at a time. Each metric/revision has separate checkpoints and an MLflow run. Rerun to resume.



In [14]:
if not GENERATION_DIRS:
    raise RuntimeError("Select generation directories first.")

for spec in METRIC_SPECS:
    pending = []
    for directory in GENERATION_DIRS:
        generated = sum(row["status"] == "ok" for row in generation_records(directory))
        scored = sum(row["status"] == "ok" for row in score_records(directory, spec))
        if scored < generated:
            pending.append(directory)

    evaluate = None
    try:
        if pending:
            try:
                evaluate = build_evaluator(spec)
            except KeyboardInterrupt:
                for directory in pending:
                    publish_safely(directory, spec, interrupted=True)
                raise
            except Exception:
                setup_error = traceback.format_exc()
                for directory in pending:
                    score_directory = directory / "scores" / fingerprint(spec)
                    write_json(score_directory / "metric.json", spec)
                    write_json(score_directory / "setup_error.json", {"error": setup_error})
                    publish_safely(directory, spec)
                warnings.warn(f"Cannot initialize {spec['name']}:\n{setup_error}")
                continue
        for directory in tqdm(GENERATION_DIRS, desc=spec["name"], unit="run"):
            interrupted = False
            try:
                score_articles(
                    directory, spec, evaluate,
                    progress=lambda rows: tqdm(
                        rows, desc="Scoring articles", unit="article", leave=False,
                    ),
                )
            except KeyboardInterrupt:
                interrupted = True
                raise
            finally:
                publish_safely(directory, spec, interrupted=interrupted)
    finally:
        del evaluate
        gc.collect()


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

[transformers] You are using a model of type `HHEMv2Config` to instantiate a model of type `HHEMv2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/113 [00:00<?, ?it/s]

hhem_consistency:   0%|          | 0/10 [00:00<?, ?run/s]

...

🏃 View run evaluation: granite4.2:3b-q4_K_M (cd3aacf4) at: http://127.0.0.1:5000/#/experiments/2/runs/dad3be9f22c74f998271f728cc0618c8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run evaluation: granite4.2:3b-q4_K_M (cd3aacf4) at: http://127.0.0.1:5000/#/experiments/2/runs/dad3be9f22c74f998271f728cc0618c8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: dad3be9f22c74f998271f728cc0618c8


Scoring articles:   0%|          | 0/100 [00:00<?, ?article/s]

C:\Users\Andakara\AppData\Local\Temp\ipykernel_215028\1430523343.py:53: UserWarning: MLflow upload failed: Cannot start run with ID 3cb2bc9ad6e54d228438caee2d7fae1e because it is in the deleted state.. Local checkpoints are safe; rerun the publication cell when the server is available.
  warnings.warn(


Scoring articles:   0%|          | 0/100 [00:00<?, ?article/s]

🏃 View run evaluation: gpt-6-luna (ee5c9dab) at: http://127.0.0.1:5000/#/experiments/2/runs/0882c53d07da48728fefc290e5544f8b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
🏃 View run evaluation: gpt-6-luna (ee5c9dab) at: http://127.0.0.1:5000/#/experiments/2/runs/0882c53d07da48728fefc290e5544f8b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLflow: 0882c53d07da48728fefc290e5544f8b


## Compare results

Reads saved results, grouped by dataset/prompt protocol. Each metric compares articles successfully scored across all selected generations.

Check `coverage`, `paired_articles`, and `excluded_articles`; use `paired_mean` for direct comparisons. No shared articles means no ranking. Repetitions appear separately, followed by averages per generator configuration.

BERTScore measures reference similarity; HHEM measures source support. Keep them separate. For close results, inspect paired differences and uncertainty before choosing a winner.


In [15]:
protocol_groups = {}
for directory in GENERATION_DIRS:
    manifest = read_json(directory / "manifest.json")
    protocol_groups.setdefault(manifest["protocol_sha256"], []).append(directory)

for protocol_id, directories in protocol_groups.items():
    for spec in METRIC_SPECS:
        rows, paired_ids = compare_generations(directories, spec)
        comparison = pd.DataFrame(rows)
        print(f"Protocol {protocol_id[:12]} | {spec['name']} | paired articles: {len(paired_ids)}")
        display(comparison)
        if paired_ids:
            display(comparison.groupby(
                ["provider", "model", "config_sha256"], dropna=False,
            ).agg(
                repetitions=("generation_id", "count"),
                paired_mean=("paired_mean", "mean"),
                paired_median_across_runs=("paired_median", "median"),
                min_run_mean=("paired_mean", "min"),
                max_run_mean=("paired_mean", "max"),
            ).sort_values("paired_mean", ascending=False))
        write_json(
            ARTIFACT_ROOT / "comparisons" / protocol_id / f"{fingerprint(spec)}.json",
            {"metric": spec, "paired_article_ids": paired_ids, "runs": rows},
        )

# Inspect low scores together with the original article and generated output.
for spec in METRIC_SPECS:
    low_scores = []
    for directory in GENERATION_DIRS:
        manifest = read_json(directory / "manifest.json")
        articles = {article["id"]: article for article in manifest["articles"]}
        outputs = {row["article_id"]: row for row in generation_records(directory)}
        for row in score_records(directory, spec):
            if row["status"] == "ok":
                low_scores.append({
                    "model": manifest["generator"]["model"],
                    "article_id": row["article_id"], "score": row["value"],
                    "source": articles[row["article_id"]]["text"],
                    "summary": outputs[row["article_id"]]["summary"],
                    "finish_reason": outputs[row["article_id"]]["metadata"].get("finish_reason"),
                })
    if low_scores:
        print(f"Lowest {spec['name']} scores")
        display(pd.DataFrame(low_scores).nsmallest(5, "score"))


Protocol 89f199ccb63e | hhem_consistency | paired articles: 100


,generation_id,provider,model,config_sha256,repeat_index,metric,expected_articles,scored_articles,coverage,paired_articles,excluded_articles,score_mean,score_median,paired_mean,paired_median,directory
0,76acfbf8b8414e908bf5f38b56eb5315,ollama,openbmb/minicpm5-2b,16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8f...,1,hhem_consistency,100,100,1.0,100,0,0.708809,0.824251,0.708809,0.824251,d:\Workspace\Repository\llm-summarization\arti...
1,7e8caf67136c4d5a87dbbef4475680b3,ollama,maternion/ling-3.0-tiny:8b-Q4_K_M,289310f68ad738b3cdeb2128b51880098e81b62c7c9154...,1,hhem_consistency,100,100,1.0,100,0,0.897178,0.938701,0.897178,0.938701,d:\Workspace\Repository\llm-summarization\arti...
2,823942191bd74154a8b0d7c7a52e3752,ollama,openbmb/minicpm5-2b,16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8f...,1,hhem_consistency,100,100,1.0,100,0,0.708809,0.824251,0.708809,0.824251,d:\Workspace\Repository\llm-summarization\arti...
3,928dcb87506246379f763a99f7b968b1,openai,gpt-6-luna,e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a8...,1,hhem_consistency,100,100,1.0,100,0,0.850074,0.915475,0.850074,0.915475,d:\Workspace\Repository\llm-summarization\arti...
4,c453a4246c324e03afce0ca112efb1cd,ollama,qwen3.5:4b-q4_K_M,97b4b60d4131e24a057e61ffba130db58ae32697d8da1e...,1,hhem_consistency,100,100,1.0,100,0,0.818621,0.896345,0.818621,0.896345,d:\Workspace\Repository\llm-summarization\arti...
5,c911ff3b41b4465eb778d215d59b64e9,ollama,LiquidAI/lfm2.5-2.6b:q4_k_m,e71c15ebc12deaf3792ce92a4ebd5bc1c917a73c90520e...,1,hhem_consistency,100,100,1.0,100,0,0.894696,0.932070,0.894696,0.932070,d:\Workspace\Repository\llm-summarization\arti...
6,cd3aacf496ce421298b5eb1e0d9a63ff,ollama,granite4.2:3b-q4_K_M,2da5d2a1a77b6b4b6d8509aa7fead2bf629ef8cad060de...,1,hhem_consistency,100,100,1.0,100,0,0.871782,0.913657,0.871782,0.913657,d:\Workspace\Repository\llm-summarization\arti...
7,e5481301c0d74e098b18bc9d4fce1459,openai,gpt-6-luna,e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a8...,1,hhem_consistency,100,100,1.0,100,0,0.866862,0.913378,0.866862,0.913378,d:\Workspace\Repository\llm-summarization\arti...
8,ee5c9dabe62d4cc38f83bd348fe1a014,openai,gpt-6-luna,a782731ac853fb1b9f6d1af590c36c6c743d7599dcb0bf...,1,hhem_consistency,100,100,1.0,100,0,0.842156,0.912336,0.842156,0.912336,d:\Workspace\Repository\llm-summarization\arti...


repetitions  \
provider model                             config_sha256                                                     
ollama   maternion/ling-3.0-tiny:8b-Q4_K_M 289310f68ad738b3cdeb2128b51880098e81b62c7c91547...            1   
         LiquidAI/lfm2.5-2.6b:q4_k_m       e71c15ebc12deaf3792ce92a4ebd5bc1c917a73c90520ed...            1   
         granite4.2:3b-q4_K_M              2da5d2a1a77b6b4b6d8509aa7fead2bf629ef8cad060de0...            1   
openai   gpt-6-luna                        e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a84...            2   
                                           a782731ac853fb1b9f6d1af590c36c6c743d7599dcb0bf9...            1   
ollama   qwen3.5:4b-q4_K_M                 97b4b60d4131e24a057e61ffba130db58ae32697d8da1ed...            1   
         openbmb/minicpm5-2b               16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8fb...            2   

                                                                                               paired_mean  \
provider model                             config_sha256                                                     
ollama   maternion/ling-3.0-tiny:8b-Q4_K_M 289310f68ad738b3cdeb2128b51880098e81b62c7c91547...     0.897178   
         LiquidAI/lfm2.5-2.6b:q4_k_m       e71c15ebc12deaf3792ce92a4ebd5bc1c917a73c90520ed...     0.894696   
         granite4.2:3b-q4_K_M              2da5d2a1a77b6b4b6d8509aa7fead2bf629ef8cad060de0...     0.871782   
openai   gpt-6-luna                        e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a84...     0.858468   
                                           a782731ac853fb1b9f6d1af590c36c6c743d7599dcb0bf9...     0.842156   
ollama   qwen3.5:4b-q4_K_M                 97b4b60d4131e24a057e61ffba130db58ae32697d8da1ed...     0.818621   
         openbmb/minicpm5-2b               16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8fb...     0.708809   

                                                                                               paired_median_across_runs  \
provider model                             config_sha256                                                                   
ollama   maternion/ling-3.0-tiny:8b-Q4_K_M 289310f68ad738b3cdeb2128b51880098e81b62c7c91547...                   0.938701   
         LiquidAI/lfm2.5-2.6b:q4_k_m       e71c15ebc12deaf3792ce92a4ebd5bc1c917a73c90520ed...                   0.932070   
         granite4.2:3b-q4_K_M              2da5d2a1a77b6b4b6d8509aa7fead2bf629ef8cad060de0...                   0.913657   
openai   gpt-6-luna                        e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a84...                   0.914427   
                                           a782731ac853fb1b9f6d1af590c36c6c743d7599dcb0bf9...                   0.912336   
ollama   qwen3.5:4b-q4_K_M                 97b4b60d4131e24a057e61ffba130db58ae32697d8da1ed...                   0.896345   
         openbmb/minicpm5-2b               16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8fb...                   0.824251   

                                                                                               min_run_mean  \
provider model                             config_sha256                                                      
ollama   maternion/ling-3.0-tiny:8b-Q4_K_M 289310f68ad738b3cdeb2128b51880098e81b62c7c91547...      0.897178   
         LiquidAI/lfm2.5-2.6b:q4_k_m       e71c15ebc12deaf3792ce92a4ebd5bc1c917a73c90520ed...      0.894696   
         granite4.2:3b-q4_K_M              2da5d2a1a77b6b4b6d8509aa7fead2bf629ef8cad060de0...      0.871782   
openai   gpt-6-luna                        e3dfb6f154689b95f81bb6bf6af420b7dc6c672b1034a84...      0.850074   
                                           a782731ac853fb1b9f6d1af590c36c6c743d7599dcb0bf9...      0.842156   
ollama   qwen3.5:4b-q4_K_M                 97b4b60d4131e24a057e61ffba130db58ae32697d8da1ed...      0.818621   
         openbmb/minicpm5-2b               16fc535555d9fdbd87efeda990f5b23d2f2c23f4e70d8fb...      0.708809 

Protocol 93ab6b013952 | hhem_consistency | paired articles: 100


,generation_id,provider,model,config_sha256,repeat_index,metric,expected_articles,scored_articles,coverage,paired_articles,excluded_articles,score_mean,score_median,paired_mean,paired_median,directory
0,a9fdec5619124b0fb8a032abd059e495,openai,gpt-6-luna,42ad85cb057a270db4790d5feaceb494e2202000bcbde0...,1,hhem_consistency,100,100,1.0,100,0,0.836344,0.912484,0.836344,0.912484,d:\Workspace\Repository\llm-summarization\arti...


,,,repetitions,paired_mean,paired_median_across_runs,min_run_mean,max_run_mean
provider,model,config_sha256,,,,,
openai,gpt-6-luna,42ad85cb057a270db4790d5feaceb494e2202000bcbde005b8e22e484916f158,1,0.836344,0.912484,0.836344,0.836344


Lowest hhem_consistency scores


,model,article_id,score,source,summary,finish_reason
68,openbmb/minicpm5-2b,dm-test-9b1af0f993d78353ebd56a1b180e882e751527b3,0.056191,Family and friends of man shot dead by police ...,A billboard honoring police officer Pablo Torr...,stop
268,openbmb/minicpm5-2b,dm-test-9b1af0f993d78353ebd56a1b180e882e751527b3,0.056191,Family and friends of man shot dead by police ...,A billboard honoring police officer Pablo Torr...,stop
26,openbmb/minicpm5-2b,dm-test-18243373494a1722ddcd162ec67b63dd749633ab,0.071767,Roma ended their four-month winless streak at ...,Roma ended a four-month home winless streak wi...,stop
226,openbmb/minicpm5-2b,dm-test-18243373494a1722ddcd162ec67b63dd749633ab,0.071767,Roma ended their four-month winless streak at ...,Roma ended a four-month home winless streak wi...,stop
926,gpt-6-luna,dm-test-18243373494a1722ddcd162ec67b63dd749633ab,0.095637,Roma ended their four-month winless streak at ...,Roma beat Napoli 1-0 to end a seven-match home...,stop


## Retry MLflow uploads

Run if uploads failed. This only syncs local checkpoints and scores. Partial runs are marked FAILED; completed retries update the same run.

In MLflow, filter `tags.run_type = 'evaluation'`, `benchmark_group`, and `metric_id`. Compare `score_mean` only with full, matching article coverage; otherwise use the local paired table.


In [16]:
# for directory in GENERATION_DIRS:
#     publish_safely(directory)
#     for spec in METRIC_SPECS:
#         if (directory / "scores" / fingerprint(spec) / "metric.json").exists():
#             publish_safely(directory, spec)
